# Campaign Deployment Overlay Example

This notebook loads the stored configuration-conditional calibration artifact, applies it to every sensor in one selected workflow campaign, and renders only the final cross-node before/after PSD overlay animation.

By default it targets the first configured testing campaign, but you can override that selection with any campaign listed under `config/notebook_workflow/`.

Deployment still uses only the persistent gain and floor laws for one sensor at a time. Any cross-sensor agreement seen in the animation is therefore a consequence of offline same-scene training, not a deployment-time coupling assumption.

The animation title keeps the campaign label, aligned record number, and the mean pairwise RMSE before and after calibration so same-scene agreement can be inspected frame by frame.


In [5]:
# Shared workflow configuration lives under ``config/notebook_workflow/``.
# Edit those files instead of maintaining notebook-local campaign lists.
NOTEBOOK_WORKFLOW_CONFIG_DIRNAME = "config/notebook_workflow"

# Optional notebook-local deployment override within the configured workflow campaigns.
# ``None`` defaults to the first configured testing campaign.
DEPLOYMENT_CAMPAIGN_LABEL = None

# Safety switch: keep the deployment notebook read-mostly unless you
# explicitly opt into fitting the lightweight bootstrap model here.
ALLOW_ARTIFACT_BOOTSTRAP_TRAINING = False


In [6]:
from __future__ import annotations

from pathlib import Path
from types import SimpleNamespace
import shutil
import sys
from tempfile import mkdtemp

from IPython.display import HTML
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import numpy as np


def resolve_repo_root() -> Path:
    """Return the repository root regardless of the current notebook cwd."""

    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "measurement_calibration").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not resolve the repository root from the notebook cwd")


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from measurement_calibration import (
    DEFAULT_ARCHIVED_ARTIFACTS_DIR,
    DEFAULT_CAMPAIGNS_DATA_DIR,
    DEFAULT_PRODUCTION_ARTIFACT_DIR,
    DEFAULT_PRODUCTION_PARAMETERS_FILENAME,
    DeploymentCalibrationResult,
    FileSystemCampaignSensorDataRepository,
    FrequencyBasisConfig,
    PersistentModelConfig,
    TwoLevelFitConfig,
    archive_artifact_directory,
    build_cross_node_campaign_animation_data,
    calibrate_sensor_observations,
    fit_and_save_calibration_corpus_model,
    fingerprint_notebook_workflow_config,
    format_cross_node_overlay_title,
    load_notebook_workflow_config,
    load_two_level_calibration_artifact,
    prepare_calibration_campaign,
    prepare_calibration_corpus,
    resolve_cross_node_overlay_limits_db,
    resolve_global_excluded_sensor_ids_by_campaign,
)

CAMPAIGNS_ROOT = REPO_ROOT / DEFAULT_CAMPAIGNS_DATA_DIR
PRODUCTION_MODEL_DIR = REPO_ROOT / DEFAULT_PRODUCTION_ARTIFACT_DIR
ARCHIVE_MODELS_ROOT = REPO_ROOT / DEFAULT_ARCHIVED_ARTIFACTS_DIR
PRODUCTION_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
WORKFLOW_CONFIG = load_notebook_workflow_config(
    REPO_ROOT / NOTEBOOK_WORKFLOW_CONFIG_DIRNAME
)
TRAINING_CAMPAIGN_LABELS = WORKFLOW_CONFIG.training_campaign_labels
TESTING_CAMPAIGN_LABELS = WORKFLOW_CONFIG.testing_campaign_labels
WORKFLOW_CAMPAIGN_LABELS = WORKFLOW_CONFIG.workflow_campaign_labels
EXCLUDED_NODES = WORKFLOW_CONFIG.excluded_sensor_ids
# Drop the same number of startup-transient rows from every retained sensor.
EXCLUDED_LEADING_MEASUREMENTS_PER_SENSOR = (
    WORKFLOW_CONFIG.excluded_leading_measurements_per_sensor
)

CAMPAIGN_REPOSITORY = FileSystemCampaignSensorDataRepository(
    campaigns_root=CAMPAIGNS_ROOT
)
AVAILABLE_CAMPAIGN_LABELS = CAMPAIGN_REPOSITORY.list_campaign_labels()
missing_campaign_labels = sorted(
    set(WORKFLOW_CAMPAIGN_LABELS).difference(AVAILABLE_CAMPAIGN_LABELS)
)
if missing_campaign_labels:
    raise ValueError(
        "Notebook workflow configuration references campaigns that do not exist "
        f"under {CAMPAIGNS_ROOT}: {missing_campaign_labels}"
    )

GLOBAL_EXCLUDED_SENSOR_IDS_BY_CAMPAIGN = resolve_global_excluded_sensor_ids_by_campaign(
    campaign_labels=WORKFLOW_CAMPAIGN_LABELS,
    excluded_sensor_ids=EXCLUDED_NODES,
    campaigns_root=CAMPAIGNS_ROOT,
)
TRAINING_EXCLUDED_SENSOR_IDS_BY_CAMPAIGN = {
    campaign_label: GLOBAL_EXCLUDED_SENSOR_IDS_BY_CAMPAIGN[campaign_label]
    for campaign_label in TRAINING_CAMPAIGN_LABELS
}
if DEPLOYMENT_CAMPAIGN_LABEL is None:
    DEPLOYMENT_CAMPAIGN_LABEL = TESTING_CAMPAIGN_LABELS[0]
elif DEPLOYMENT_CAMPAIGN_LABEL not in WORKFLOW_CAMPAIGN_LABELS:
    raise ValueError(
        "DEPLOYMENT_CAMPAIGN_LABEL must belong to the configured workflow campaigns, "
        f"got {DEPLOYMENT_CAMPAIGN_LABEL!r} not in {WORKFLOW_CAMPAIGN_LABELS}"
    )

BASIS_CONFIG = FrequencyBasisConfig(
    n_gain_basis=10,
    n_floor_basis=8,
    n_variance_basis=8,
    spline_degree=3,
)
MODEL_CONFIG = PersistentModelConfig(
    sensor_embedding_dim=4,
    configuration_latent_dim=4,
)
# Keep bootstrap training consistent with the main notebook. Because the
# consistency loss is averaged while the Gaussian likelihood is summed, this
# checked-in value is only a light regularizer; visibly stronger same-scene RMSE
# pressure typically requires much larger lambda_consistency values.
FIT_CONFIG = TwoLevelFitConfig(
    n_outer_iterations=8,
    n_gradient_steps=20,
    learning_rate=0.03,
    sigma_min=1.0e-8,
    adaptive_variance_floor_ratio=1.0e-4,
    lambda_consistency=0.05,
    consistency_log_floor_power=1.0e-10,
    select_best_outer_iterate=True,
    early_stopping_patience=5,
    divergence_tolerance_ratio=10.0,
    refresh_campaign_variance_from_residuals=True,
    variance_refresh_ridge=1.0e-6,
)


## Load The Stored Model

By default this notebook expects a promoted artifact under `models/production/` and does not retrain automatically. Set `ALLOW_ARTIFACT_BOOTSTRAP_TRAINING = True` only if you intentionally want this notebook to fit the lightweight demo configuration and promote it into production.


In [7]:
if not (PRODUCTION_MODEL_DIR / "manifest.json").exists():
    if not ALLOW_ARTIFACT_BOOTSTRAP_TRAINING:
        raise FileNotFoundError(
            "No promoted artifact exists under models/production/. "
            "Run the training notebook intentionally, or set "
            "ALLOW_ARTIFACT_BOOTSTRAP_TRAINING = True here if you really want "
            "this deployment notebook to build the lightweight demo model."
        )
    staging_output_dir = Path(
        mkdtemp(prefix=".production_staging__", dir=PRODUCTION_MODEL_DIR.parent)
    )
    try:
        preparation = prepare_calibration_corpus(
            campaign_labels=TRAINING_CAMPAIGN_LABELS,
            campaigns_root=CAMPAIGNS_ROOT,
            excluded_sensor_ids_by_campaign=TRAINING_EXCLUDED_SENSOR_IDS_BY_CAMPAIGN,
            excluded_leading_measurements_per_sensor=(
                EXCLUDED_LEADING_MEASUREMENTS_PER_SENSOR
            ),
        )
        fit_and_save_calibration_corpus_model(
            preparation=preparation,
            output_dir=staging_output_dir,
            basis_config=BASIS_CONFIG,
            model_config=MODEL_CONFIG,
            fit_config=FIT_CONFIG,
            parameters_filename=DEFAULT_PRODUCTION_PARAMETERS_FILENAME,
        )
        archive_artifact_directory(
            output_dir=PRODUCTION_MODEL_DIR,
            archive_root=ARCHIVE_MODELS_ROOT,
            archive_label="production",
        )
        if PRODUCTION_MODEL_DIR.exists():
            PRODUCTION_MODEL_DIR.rmdir()
        staging_output_dir.rename(PRODUCTION_MODEL_DIR)
    except Exception:
        shutil.rmtree(staging_output_dir, ignore_errors=True)
        raise

artifact = load_two_level_calibration_artifact(PRODUCTION_MODEL_DIR)
artifact_workflow_fingerprint = artifact.manifest.get("provenance", {}).get(
    "workflow_config_fingerprint"
)
current_workflow_fingerprint = fingerprint_notebook_workflow_config(
    REPO_ROOT / NOTEBOOK_WORKFLOW_CONFIG_DIRNAME
)
if artifact_workflow_fingerprint not in {None, current_workflow_fingerprint}:
    print(
        "Note: the promoted production artifact predates the current notebook "
        "workflow files. Deployment remains usable, but retraining is still "
        "needed to restore full workflow/artifact reproducibility."
    )
deployment_excluded_sensor_ids = GLOBAL_EXCLUDED_SENSOR_IDS_BY_CAMPAIGN[
    DEPLOYMENT_CAMPAIGN_LABEL
]
deployment_campaign = prepare_calibration_campaign(
    campaign_label=DEPLOYMENT_CAMPAIGN_LABEL,
    campaigns_root=CAMPAIGNS_ROOT,
    excluded_sensor_ids=deployment_excluded_sensor_ids,
    excluded_leading_measurements_per_sensor=(
        EXCLUDED_LEADING_MEASUREMENTS_PER_SENSOR
    ),
)


## Cross-Node Overlay Animation

This notebook intentionally emits only the final before/after campaign overlay animation for the selected deployment campaign.

The overlay uses deployed persistent calibration curves only. The same-scene assumption is not reintroduced at deployment time; it only appears indirectly through the artifact that was trained offline.

Each frame overlays every retained sensor on the full frequency grid, with the title showing the campaign label, aligned record number, and the mean pairwise RMSE before and after calibration.


In [8]:
def calibrate_campaign_sensor(
    sensor_id: str,  # Sensor to calibrate
    observations_power: np.ndarray,  # Sensor PSD tensor [linear power]
) -> DeploymentCalibrationResult:  # Calibrated deployment output
    """Bind the deployment artifact and campaign metadata to one sensor."""

    return calibrate_sensor_observations(
        result=artifact.result,
        sensor_id=sensor_id,
        configuration=deployment_campaign.campaign.configuration,
        frequency_hz=deployment_campaign.campaign.frequency_hz,
        observations_power=observations_power,
    )


def build_campaign_overlay_data(
    campaign,  # Same-scene campaign used only for visualization
    calibrate_sensor,  # Deployment-bound single-sensor calibrator
):
    """Build a notebook-friendly overlay payload from the shared helper."""

    cross_node_animation_data = build_cross_node_campaign_animation_data(
        campaign=campaign,
        calibrate_sensor=calibrate_sensor,
    )
    return SimpleNamespace(
        campaign_label=cross_node_animation_data.campaign_label,
        sensor_ids=cross_node_animation_data.sensor_ids,
        frequency_hz=cross_node_animation_data.frequency_hz,
        raw_power_db=cross_node_animation_data.raw_power_db,
        calibrated_power_db=cross_node_animation_data.calibrated_power_db,
        record_alignments=cross_node_animation_data.record_alignments,
        frame_alignment_rows=[
            {
                "record_index": float(alignment.record_number),
                "mean_pairwise_raw_rmse_db": float(
                    alignment.mean_pairwise_raw_rmse_db
                ),
                "mean_pairwise_calibrated_rmse_db": float(
                    alignment.mean_pairwise_calibrated_rmse_db
                ),
            }
            for alignment in cross_node_animation_data.record_alignments
        ],
        n_records=cross_node_animation_data.n_records,
        n_sensors=cross_node_animation_data.n_sensors,
    )


overlay_data = build_campaign_overlay_data(
    campaign=deployment_campaign.campaign,
    calibrate_sensor=calibrate_campaign_sensor,
)
animation_data = overlay_data
frequency_mhz = overlay_data.frequency_hz / 1.0e6
y_min_db, y_max_db = resolve_cross_node_overlay_limits_db(overlay_data)

fig, (raw_axis, calibrated_axis) = plt.subplots(
    2,
    1,
    figsize=(14, 8.5),
    sharex=True,
    constrained_layout=True,
)

line_handles = {}
color_map = plt.get_cmap("tab10")
for sensor_index, sensor_id in enumerate(overlay_data.sensor_ids):
    color = color_map(sensor_index % color_map.N)
    raw_line, = raw_axis.plot([], [], linewidth=1.0, alpha=0.9, color=color, label=sensor_id)
    calibrated_line, = calibrated_axis.plot(
        [],
        [],
        linewidth=1.0,
        alpha=0.9,
        color=color,
        label=sensor_id,
    )
    line_handles[sensor_id] = (raw_line, calibrated_line)

# Keep both panels on the same scale so the visual compression is
# directly comparable before and after calibration.
for axis, axis_title in (
    (raw_axis, "Before Calibration"),
    (calibrated_axis, "After Calibration"),
):
    axis.set_xlim(frequency_mhz[0], frequency_mhz[-1])
    axis.set_ylim(y_min_db, y_max_db)
    axis.set_ylabel("Power [dB]")
    axis.set_title(axis_title)
    axis.grid(alpha=0.3)

calibrated_axis.set_xlabel("Frequency [MHz]")
raw_axis.legend(loc="upper right", ncols=min(4, overlay_data.n_sensors))


def update_animation(record_index: int):
    """Update both overlay panels for one aligned record."""

    artists = []
    for sensor_index, sensor_id in enumerate(overlay_data.sensor_ids):
        raw_line, calibrated_line = line_handles[sensor_id]
        raw_line.set_data(
            frequency_mhz,
            overlay_data.raw_power_db[sensor_index, record_index],
        )
        calibrated_line.set_data(
            frequency_mhz,
            overlay_data.calibrated_power_db[sensor_index, record_index],
        )
        artists.extend((raw_line, calibrated_line))

    fig.suptitle(
        format_cross_node_overlay_title(animation_data, record_index),
        fontsize=14,
    )
    return artists


animation = FuncAnimation(
    fig,
    update_animation,
    frames=overlay_data.n_records,
    interval=250,
    blit=False,
    cache_frame_data=False,
)
plt.close(fig)
overlay_animation = HTML(animation.to_jshtml(default_mode="loop"))
overlay_animation
